In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display all columns
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [ ]:
BASE_DIR = Path("../")

RAW_DATA = BASE_DIR / "data" / "raw"
CLEANED_DATA = BASE_DIR / "data" / "cleaned"

In [ ]:
customers = pd.read_csv(
    RAW_DATA / "olist_customers_dataset.csv"
)

In [ ]:
customers.head()

In [ ]:
customers.sample(5)

In [ ]:
customers.shape

In [ ]:
customers.isnull().sum()

In [ ]:
customers.duplicated().sum()

In [ ]:
customers["customer_id"].duplicated().sum()

In [ ]:
customers["customer_unique_id"].duplicated().sum()

In [ ]:
profile = pd.DataFrame({
    "Column": customers.columns,
    "Data Type": customers.dtypes.values,
    "Missing Values": customers.isnull().sum().values,
    "Missing %": (
        customers.isnull().sum() / len(customers) * 100
    ).round(2).values,
    "Unique Values": customers.nunique().values
})

profile

In [ ]:
customers["customer_state"].unique()

In [ ]:
missing = (
    customers.isnull().mean() * 100
).round(2)

missing

In [ ]:
customers["customer_state"].value_counts()

In [ ]:
customers["customer_state"] = (
    customers["customer_state"]
    .str.upper()
    .str.strip()
)

In [ ]:
customers["customer_city"] = (
    customers["customer_city"]
    .str.strip()
    .str.lower()
)

In [ ]:
customers["customer_zip_code_prefix"].describe()

In [ ]:
(customers["customer_zip_code_prefix"] <= 0).sum()

In [ ]:
customers.info()

In [ ]:
customers.duplicated().sum()

In [ ]:
customers.to_csv(
    CLEANED_DATA / "customers_cleaned.csv",
    index=False
)

In [ ]:
orders = pd.read_csv(RAW_DATA / "olist_orders_dataset.csv")

In [ ]:
orders.head()
orders.sample(5)
orders.shape
orders.info()
orders.describe(include="all")

In [ ]:
profile = pd.DataFrame({
    "Column": orders.columns,
    "Data Type": orders.dtypes.values,
    "Missing": orders.isnull().sum().values,
    "Missing %": (
        orders.isnull().mean()*100
    ).round(2).values,
    "Unique": orders.nunique().values
})

profile

In [ ]:
orders.isnull().sum()

In [ ]:
orders["order_status"].value_counts()

In [ ]:
cancelled = orders[
    orders["order_status"]=="canceled"
]

cancelled.isnull().sum()

In [ ]:
orders.duplicated().sum()

In [ ]:
orders["order_id"].duplicated().sum()

In [ ]:
missing_customers = (
    ~orders["customer_id"].isin(
        customers["customer_id"]
    )
).sum()

print(missing_customers)

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [ ]:
orders.info()

In [ ]:
invalid_delivery = orders[
    orders["order_delivered_customer_date"]
    <
    orders["order_purchase_timestamp"]
]

len(invalid_delivery)

In [ ]:
invalid_approval = orders[
    orders["order_approved_at"]
    <
    orders["order_purchase_timestamp"]
]

len(invalid_approval)

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    -
    orders["order_purchase_timestamp"]
).dt.days

In [ ]:
orders["delivery_days"].describe()

In [ ]:
orders["late_delivery"] = (
    orders["order_delivered_customer_date"]
    >
    orders["order_estimated_delivery_date"]
)

In [ ]:
orders["late_delivery"].value_counts()

In [ ]:
orders.to_csv(
    CLEANED_DATA/"orders_cleaned.csv",
    index=False
)

In [ ]:
products = pd.read_csv(
    RAW_DATA / "olist_products_dataset.csv"
)

# Standardize column names
products.rename(
    columns={
        "product_name_lenght":"product_name_length",
        "product_description_lenght":"product_description_length"
    },
    inplace=True
)

In [ ]:
products.head()
products.sample(5)
products.shape
products.info()
products.describe(include="all")

In [ ]:
profile = pd.DataFrame({

"Column":products.columns,

"Datatype":products.dtypes.values,

"Missing":products.isnull().sum().values,

"Missing %":(
products.isnull().mean()*100
).round(2).values,

"Unique":products.nunique().values

})

profile

In [ ]:
products.isnull().sum()

In [ ]:
missing_products = products[
    products.isnull().any(axis=1)
]

missing_products.head()

In [ ]:
products["product_category_name"].isnull().sum()

In [ ]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("Unknown")
)

In [ ]:
products["product_name_length"].isnull().sum()

In [ ]:
products["product_name_length"] = (

products["product_name_length"]

.fillna(

products["product_name_length"].median()

)

)

In [ ]:
products["product_name_length"] = (

products["product_name_length"]

.fillna(

products["product_name_length"].median()

)

)

In [ ]:
products["product_photos_qty"] = (

products["product_photos_qty"]

.fillna(

products["product_photos_qty"].median()

)

)

In [ ]:
products["product_weight_g"]=(

products["product_weight_g"]

.fillna(

products["product_weight_g"].median()

)

)

In [ ]:
dimension_cols=[

"product_length_cm",

"product_height_cm",

"product_width_cm"

]

for col in dimension_cols:

    products[col]=(

    products[col]

    .fillna(

    products[col].median()

    )

)

In [ ]:
products.dtypes

In [ ]:
int_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty"
]

for col in int_columns:
    # Notice the capital 'I' and the quotes around 'Int64'
    products[col] = products[col].astype('Int64')

In [ ]:
products.duplicated().sum()

In [ ]:
products["product_id"].duplicated().sum()

In [ ]:
(products["product_weight_g"]<=0).sum()

In [ ]:
products[products["product_weight_g"] <= 0][
    ["product_id", "product_weight_g"]
]

In [ ]:
median_weight = products.loc[
    products["product_weight_g"] > 0,
    "product_weight_g"
].median()

products.loc[
    products["product_weight_g"] <= 0,
    "product_weight_g"
] = median_weight

In [ ]:
(products["product_length_cm"]<=0).sum()

In [ ]:
(products["product_height_cm"]<=0).sum()

In [ ]:
(products["product_width_cm"]<=0).sum()

In [ ]:
products["product_category_name"].value_counts().head(20)

In [ ]:
products["product_volume_cm3"]=(

products["product_length_cm"]

*

products["product_height_cm"]

*

products["product_width_cm"]

)

In [ ]:
products["density"]=(

products["product_weight_g"]

/

products["product_volume_cm3"]

)

In [ ]:
numeric_cols=[

"product_weight_g",

"product_volume_cm3",

"density"

]

products[numeric_cols].describe()

In [ ]:
products.info()

In [ ]:
products.to_csv(

CLEANED_DATA/

"products_cleaned.csv",

index=False

)

In [ ]:
Q1 = products["product_weight_g"].quantile(0.25)
Q3 = products["product_weight_g"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = products[
    (products["product_weight_g"] < lower) |
    (products["product_weight_g"] > upper)
]

print(f"Number of Outliers: {len(outliers)}")

In [ ]:
sellers = pd.read_csv(
    RAW_DATA / "olist_sellers_dataset.csv"
)

In [ ]:
sellers.head()

In [ ]:
sellers.info()

In [ ]:
sellers.describe(include="all")

In [ ]:
sellers.shape

In [ ]:
profile = pd.DataFrame({
    "Column": sellers.columns,
    "Data Type": sellers.dtypes.values,
    "Missing": sellers.isnull().sum().values,
    "Missing %": (
        sellers.isnull().mean() * 100
    ).round(2).values,
    "Unique": sellers.nunique().values
})

profile

In [ ]:
sellers.isnull().sum()

In [ ]:
sellers.duplicated().sum()

In [ ]:
sellers["seller_id"].duplicated().sum()

In [ ]:
sellers["seller_city"] = (
    sellers["seller_city"]
    .str.lower()
    .str.strip()
)

sellers["seller_state"] = (
    sellers["seller_state"]
    .str.upper()
    .str.strip()
)

In [ ]:
(sellers["seller_zip_code_prefix"] <= 0).sum()

In [ ]:
sellers["seller_state"].value_counts()

In [ ]:
sellers.to_csv(
    CLEANED_DATA / "sellers_cleaned.csv",
    index=False
)

In [ ]:
order_items = pd.read_csv(
    RAW_DATA/"olist_order_items_dataset.csv"
)

In [ ]:
order_items.head()

In [ ]:
order_items.info()

In [ ]:
order_items.describe(include="all")

In [ ]:
order_items.shape

In [ ]:
profile = pd.DataFrame({

"Column":order_items.columns,

"Datatype":order_items.dtypes.values,

"Missing":order_items.isnull().sum().values,

"Missing %":(
order_items.isnull().mean()*100
).round(2).values,

"Unique":order_items.nunique().values

})

profile

In [ ]:
order_items.isnull().sum()

In [ ]:
order_items.duplicated().sum()

In [ ]:
order_items[
["order_id","order_item_id"]
].duplicated().sum()

In [ ]:
(~order_items["product_id"].isin(
products["product_id"]
)).sum()

In [ ]:
(~order_items["seller_id"].isin(
sellers["seller_id"]
)).sum()

In [ ]:
(~order_items["order_id"].isin(
orders["order_id"]
)).sum()

In [ ]:
order_items["shipping_limit_date"]=pd.to_datetime(

order_items["shipping_limit_date"]

)

In [ ]:
(order_items["price"]<=0).sum()

In [ ]:
(order_items["freight_value"]<0).sum()

In [ ]:
order_items["revenue"]=order_items["price"]

In [ ]:
order_items["freight_percentage"]=(
order_items["freight_value"]
/
order_items["price"]
)*100

In [ ]:
order_value=order_items.groupby(

"order_id"

)["price"].sum()

In [ ]:
order_items=order_items.merge(

order_value.rename("order_value"),

on="order_id"

)

In [ ]:
order_items=order_items.merge(

order_value.rename("order_value"),

on="order_id"

)

In [ ]:
basket=order_items.groupby(

"order_id"

).size()

In [ ]:
order_items=order_items.merge(

basket.rename("basket_size"),

on="order_id"

)

In [ ]:
seller_rev=order_items.groupby(

"seller_id"

)["price"].sum()

In [ ]:
order_items=order_items.merge(

seller_rev.rename("seller_revenue"),

on="seller_id"

)

In [ ]:
product_rev=order_items.groupby(

"product_id"

)["price"].sum()

In [ ]:
order_items=order_items.merge(

product_rev.rename("product_revenue"),

on="product_id"

)

In [ ]:
avg_price=order_items.groupby(

"product_id"

)["price"].mean()

In [ ]:
order_items=order_items.merge(

avg_price.rename(

"avg_product_price"

),

on="product_id"

)

In [ ]:
avg_freight=order_items.groupby(

"seller_id"

)["freight_value"].mean()

In [ ]:
order_items=order_items.merge(

avg_freight.rename(

"avg_seller_freight"

),

on="seller_id"

)

In [ ]:
order_items["price"].describe()

In [ ]:
order_items["freight_value"].describe()

In [ ]:
order_items.to_csv(

CLEANED_DATA/

"order_items_cleaned.csv",

index=False
)

In [ ]:
payments = pd.read_csv(
    RAW_DATA/"olist_order_payments_dataset.csv"
)

In [ ]:
payments.head()
payments.info()
payments.describe(include="all")
payments.shape

In [ ]:
payments.isnull().sum()

In [ ]:
payments.duplicated().sum()

In [ ]:
payments[
["order_id","payment_sequential"]
].duplicated().sum()

In [ ]:
payments["payment_type"].value_counts()

In [ ]:
payments["payment_type"] = (

payments["payment_type"]

.str.lower()

.str.strip()

)

In [ ]:
payments["payment_installments"].describe()

In [ ]:
(payments["payment_installments"]<=0).sum()

In [ ]:
payments["payment_value"].describe()

In [ ]:
payments.to_csv(

CLEANED_DATA/

"payments_cleaned.csv",

index=False

)

In [ ]:
reviews = pd.read_csv(
RAW_DATA/"olist_order_reviews_dataset.csv"
)

In [ ]:
reviews.info()

reviews.describe(include="all")

reviews.shape

In [ ]:
reviews.isnull().sum()

In [ ]:
reviews.isnull().sum()

In [ ]:
reviews["review_score"].value_counts()

In [ ]:
(~reviews["review_score"].isin([1,2,3,4,5])).sum()

In [ ]:
reviews["order_id"].duplicated().sum()

In [ ]:
reviews["review_creation_date"] = pd.to_datetime(
reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
reviews["review_answer_timestamp"]
)

In [ ]:
reviews.to_csv(

CLEANED_DATA/

"reviews_cleaned.csv",

index=False

)

In [ ]:
geo = pd.read_csv(
RAW_DATA/"olist_geolocation_dataset.csv"
)

In [ ]:
geo.isnull().sum()

In [ ]:
geo["geolocation_lat"].between(-90,90).all()

In [ ]:
geo["geolocation_lng"].between(-180,180).all()

In [ ]:
geo["geolocation_city"] = (
    geo["geolocation_city"]
    .str.lower()
    .str.strip()
)

geo["geolocation_state"] = (
    geo["geolocation_state"]
    .str.upper()
    .str.strip()
)

In [ ]:
geo.duplicated().sum()

In [ ]:
geo.to_csv(

CLEANED_DATA/

"geolocation_cleaned.csv",

index=False

)

In [ ]:
category = pd.read_csv(
RAW_DATA/"product_category_name_translation.csv"
)

In [ ]:
category.info()

category.describe(include="all")

In [ ]:
category.isnull().sum()

In [ ]:
category.duplicated().sum()

In [ ]:
category["product_category_name"].duplicated().sum()

In [ ]:
category["product_category_name"] = (
    category["product_category_name"]
    .str.lower()
    .str.strip()
)

category["product_category_name_english"] = (
    category["product_category_name_english"]
    .str.lower()
    .str.strip()
)

In [ ]:
category["product_category_name"] = (
    category["product_category_name"]
    .str.lower()
    .str.strip()
)

category["product_category_name_english"] = (
    category["product_category_name_english"]
    .str.lower()
    .str.strip()
)

In [ ]:
category.to_csv(

CLEANED_DATA/

"category_translation_cleaned.csv",

index=False

)